# RetailFlow — Local ML Demand Forecasting

**Purpose:** Train and evaluate a local scikit-learn demand model and produce a stable recursive 7-day forecast without Spark ML inference.

### Architecture

`MinIO/Delta → Spark (read once) → Pandas → scikit-learn → recursive forecast → summaries`

Spark is used only to read the existing Gold forecasting dataset. After the one-time `toPandas()` transfer, all feature engineering, training, validation, recursive inference, and summaries run locally.

**Important:** The future forecast produced here is a genuine ML forecast from the local Gradient Boosting model. It is not the weighted `0.4 * lag_1 + 0.6 * lag_7` fallback.


In [1]:
# ============================================================
# CELL 1 — ENVIRONMENT
# ============================================================

import os
import sys
from pathlib import Path
from datetime import date, timedelta

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Python :", sys.version.split()[0])
print("Project:", PROJECT_ROOT)


Python : 3.11.5
Project: c:\RetailFlow


In [2]:
%pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from pyspark.sql import functions as F

from core.constants import GOLD_FORECASTING_DATASET_PATH
from core.spark_session import SparkSessionManager

print("scikit-learn:", sklearn.__version__)
print("Pandas      :", pd.__version__)
print("NumPy       :", np.__version__)


scikit-learn: 1.9.0
Pandas      : 2.3.1
NumPy       : 2.3.1


In [4]:
# ============================================================
# CELL 3 — USE THE EXISTING CENTRALIZED SPARK SESSION
# ============================================================

# Do NOT create another SparkSession here.
# The project already provides the centralized Spark session manager.

spark = SparkSessionManager.get_session()
spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)


2026-08-18 18:38:53 | INFO     | core.spark_session:get_session:36 | Creating Spark Session...
2026-08-18 18:38:53 | INFO     | core.spark_session:get_session:47 | Detected local Windows environment.
2026-08-18 18:38:53 | INFO     | core.spark_session:get_session:61 | Connecting Hadoop S3A FileSystem to endpoint: localhost:9000
2026-08-18 18:39:02 | SUCCESS  | core.spark_session:get_session:167 | Spark Session created successfully.
Spark: 3.5.1
Master: local[*]


In [5]:
# ============================================================
# CELL 4 — LOAD GOLD DATASET ONCE
# ============================================================

print("Loading:", GOLD_FORECASTING_DATASET_PATH)

forecasting_spark_df = (
    spark.read
    .format("delta")
    .load(GOLD_FORECASTING_DATASET_PATH)
)

print("Spark dataset loaded.")
print("Columns:", len(forecasting_spark_df.columns))
forecasting_spark_df.printSchema()


Loading: s3a://retailflow/gold/forecasting_dataset
Spark dataset loaded.
Columns: 23
root
 |-- date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- quantity_sold: long (nullable = true)
 |-- revenue_cents: long (nullable = true)
 |-- transactions: long (nullable = true)
 |-- average_unit_price_cents: decimal(20,2) (nullable = true)
 |-- discount_cents: long (nullable = true)
 |-- tax_cents: long (nullable = true)
 |-- promotion_applied: integer (nullable = true)
 |-- promotion_transactions: long (nullable = true)
 |-- inventory_min_before: long (nullable = true)
 |-- inventory_end: long (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- day_of_month: integer (nullab

In [6]:
# ============================================================
# CELL 5 — LOAD HISTORICAL DATA INTO PANDAS
# ============================================================

print("Loading historical forecasting data into Pandas...")

history_pdf = (
    forecasting_spark_df
    .select(
        "date",
        "store_id",
        "product_id",
        "quantity_sold",
        "promotion_applied",
        "promotion_transactions",
        "inventory_min_before",
        "average_unit_price_cents",
    )
    .toPandas()
)

print("=" * 60)
print("PANDAS DATASET LOADED")
print("=" * 60)
print("Rows   :", len(history_pdf))
print("Columns:", len(history_pdf.columns))
print()
print(history_pdf.head())

assert len(history_pdf) == 9000, (
    f"Expected 9000 rows, got {len(history_pdf)}"
)

print("\nHistorical Pandas dataset: PASSED")

Loading historical forecasting data into Pandas...
PANDAS DATASET LOADED
Rows   : 9000
Columns: 8

         date store_id product_id  quantity_sold  promotion_applied  \
0  2026-05-23   STR004    PRD0009              4                  1   
1  2026-05-23   STR002    PRD0001              5                  1   
2  2026-05-23   STR001    PRD0007              1                  0   
3  2026-02-17   STR003    PRD0009              2                  0   
4  2026-07-17   STR003    PRD0006              3                  0   

   promotion_transactions  inventory_min_before average_unit_price_cents  
0                       1                    17                149900.00  
1                       1                    32                  6800.00  
2                       0                     2               2999000.00  
3                       0                   106                149900.00  
4                       0                    68                199900.00  

Historical Pandas datas

In [7]:
# ============================================================
# CELL 6 — DATA VALIDATION IN PANDAS
# ============================================================

history_pdf["date"] = pd.to_datetime(history_pdf["date"]).dt.date
history_pdf["quantity_sold"] = pd.to_numeric(
    history_pdf["quantity_sold"], errors="raise"
).astype(float)

for col in [
    "promotion_applied",
    "promotion_transactions",
    "inventory_min_before",
    "average_unit_price_cents",
]:
    history_pdf[col] = pd.to_numeric(history_pdf[col], errors="coerce")

history_pdf = history_pdf.sort_values(
    ["store_id", "product_id", "date"]
).reset_index(drop=True)

EXPECTED_PAIRS = 50
EXPECTED_DAYS = 180

rows = len(history_pdf)
pairs = history_pdf[["store_id", "product_id"]].drop_duplicates().shape[0]
grain = history_pdf[
    ["date", "store_id", "product_id"]
].drop_duplicates().shape[0]

min_date = history_pdf["date"].min()
max_date = history_pdf["date"].max()

print("=" * 60)
print("LOCAL DATASET VALIDATION")
print("=" * 60)
print("Rows                 :", rows)
print("Store-product pairs  :", pairs)
print("Grain rows           :", grain)
print("Duplicate grain rows :", rows - grain)
print("Date range           :", min_date, "->", max_date)

assert rows == 9000
assert pairs == EXPECTED_PAIRS
assert rows == grain
assert max_date == date(2026, 8, 12)

print("Validation: PASSED")


LOCAL DATASET VALIDATION
Rows                 : 9000
Store-product pairs  : 50
Grain rows           : 9000
Duplicate grain rows : 0
Date range           : 2026-02-14 -> 2026-08-12
Validation: PASSED


In [8]:
# ============================================================
# CELL 7 — FEATURE ENGINEERING
# ============================================================

GROUP_COLS = ["store_id", "product_id"]

feature_pdf = history_pdf.copy()

grouped = feature_pdf.groupby(GROUP_COLS, sort=False)["quantity_sold"]

feature_pdf["lag_1"] = grouped.shift(1)
feature_pdf["lag_7"] = grouped.shift(7)
feature_pdf["lag_14"] = grouped.shift(14)
feature_pdf["lag_28"] = grouped.shift(28)

feature_pdf["rolling_mean_7"] = (
    grouped
    .rolling(7)
    .mean()
    .reset_index(level=GROUP_COLS, drop=True)
    .shift(1)
)

feature_pdf["rolling_mean_14"] = (
    grouped
    .rolling(14)
    .mean()
    .reset_index(level=GROUP_COLS, drop=True)
    .shift(1)
)

feature_pdf["rolling_mean_28"] = (
    grouped
    .rolling(28)
    .mean()
    .reset_index(level=GROUP_COLS, drop=True)
    .shift(1)
)

feature_pdf["day_of_week"] = pd.to_datetime(feature_pdf["date"]).dt.dayofweek
feature_pdf["day_of_month"] = pd.to_datetime(feature_pdf["date"]).dt.day
feature_pdf["month"] = pd.to_datetime(feature_pdf["date"]).dt.month

MODEL_FEATURES = [
    "store_id",
    "product_id",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "promotion_applied",
    "promotion_transactions",
    "inventory_min_before",
    "average_unit_price_cents",
    "day_of_week",
    "day_of_month",
    "month",
]

print("Feature engineering completed.")
print("Rows:", len(feature_pdf))


Feature engineering completed.
Rows: 9000


In [9]:
# ============================================================
# CELL 8 — FEATURE VALIDATION / LEAKAGE CHECK
# ============================================================

# Verify that rolling features use only previous observations.
series = (
    feature_pdf[
        (feature_pdf["store_id"] == "STR001") &
        (feature_pdf["product_id"] == "PRD0001")
    ]
    .sort_values("date")
    .reset_index(drop=True)
)

day29 = series.iloc[28]
manual_mean_7 = series.loc[21:27, "quantity_sold"].mean()

print("Day 29:", day29["date"])
print("Spark/Pandas rolling_mean_7:", day29["rolling_mean_7"])
print("Manual previous-7 mean    :", manual_mean_7)

assert np.isclose(
    day29["rolling_mean_7"],
    manual_mean_7,
    atol=1e-9,
)

model_pdf = feature_pdf.dropna(
    subset=[
        "lag_1", "lag_7", "lag_14", "lag_28",
        "rolling_mean_7", "rolling_mean_14", "rolling_mean_28"
    ]
).copy()

print("Model-ready rows:", len(model_pdf))
print("Model-ready range:", model_pdf["date"].min(), "->", model_pdf["date"].max())

assert len(model_pdf) == 7600
assert model_pdf["date"].min() == date(2026, 3, 14)

print("Feature validation: PASSED")


Day 29: 2026-03-14
Spark/Pandas rolling_mean_7: 11.285714285714286
Manual previous-7 mean    : 11.285714285714286
Model-ready rows: 7600
Model-ready range: 2026-03-14 -> 2026-08-12
Feature validation: PASSED


In [10]:
# ============================================================
# CELL 9 — CHRONOLOGICAL TRAIN / TEST SPLIT
# ============================================================

TEST_DAYS = 14

last_model_date = model_pdf["date"].max()
test_start = last_model_date - timedelta(days=TEST_DAYS - 1)

train_pdf = model_pdf[model_pdf["date"] < test_start].copy()
test_pdf = model_pdf[model_pdf["date"] >= test_start].copy()

print("=" * 60)
print("CHRONOLOGICAL SPLIT")
print("=" * 60)
print("Train rows :", len(train_pdf))
print("Test rows  :", len(test_pdf))
print("Train range:", train_pdf["date"].min(), "->", train_pdf["date"].max())
print("Test range :", test_pdf["date"].min(), "->", test_pdf["date"].max())

assert len(train_pdf) == 6900
assert len(test_pdf) == 700
assert train_pdf["date"].max() < test_pdf["date"].min()

print("Split validation: PASSED")


CHRONOLOGICAL SPLIT
Train rows : 6900
Test rows  : 700
Train range: 2026-03-14 -> 2026-07-29
Test range : 2026-07-30 -> 2026-08-12
Split validation: PASSED


In [11]:
# ============================================================
# CELL 10 — LOCAL CATEGORICAL ENCODING + FEATURE MATRIX
# ============================================================

CATEGORICAL_FEATURES = ["store_id", "product_id"]

NUMERIC_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "promotion_applied",
    "promotion_transactions",
    "inventory_min_before",
    "average_unit_price_cents",
    "day_of_week",
    "day_of_month",
    "month",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            CATEGORICAL_FEATURES,
        ),
        ("numeric", "passthrough", NUMERIC_FEATURES),
    ],
    remainder="drop",
)

X_train = preprocessor.fit_transform(train_pdf[MODEL_FEATURES])
X_test = preprocessor.transform(test_pdf[MODEL_FEATURES])

y_train = train_pdf["quantity_sold"].to_numpy(dtype=float)
y_test = test_pdf["quantity_sold"].to_numpy(dtype=float)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


X_train: (6900, 29)
X_test : (700, 29)
y_train: (6900,)
y_test : (700,)


In [12]:
# ============================================================
# CELL 11 — TRAIN LOCAL GRADIENT BOOSTING MODEL
# ============================================================

local_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    random_state=42,
    loss="squared_error",
)

print("Training local GradientBoostingRegressor...")

local_model.fit(X_train, y_train)

print("Local ML model trained successfully.")


Training local GradientBoostingRegressor...
Local ML model trained successfully.


In [13]:
# ============================================================
# CELL 12 — HISTORICAL BACKTEST
# ============================================================

test_predictions = np.maximum(
    local_model.predict(X_test),
    0.0,
)

backtest_pdf = test_pdf[
    ["date", "store_id", "product_id", "quantity_sold", "lag_7"]
].copy()

backtest_pdf["prediction"] = test_predictions
backtest_pdf["baseline_prediction"] = backtest_pdf["lag_7"]

def calculate_metrics(actual, prediction):
    actual = np.asarray(actual, dtype=float)
    prediction = np.asarray(prediction, dtype=float)

    absolute_error = np.abs(actual - prediction)

    mae = float(np.mean(absolute_error))
    rmse = float(np.sqrt(np.mean((actual - prediction) ** 2)))

    denominator = np.sum(np.abs(actual))
    wape = (
        float(np.sum(absolute_error) / denominator * 100)
        if denominator != 0
        else np.nan
    )

    return mae, rmse, wape

local_mae, local_rmse, local_wape = calculate_metrics(
    backtest_pdf["quantity_sold"],
    backtest_pdf["prediction"],
)

baseline_mae, baseline_rmse, baseline_wape = calculate_metrics(
    backtest_pdf["quantity_sold"],
    backtest_pdf["baseline_prediction"],
)

print("=" * 70)
print("LOCAL ML MODEL COMPARISON — 14-DAY BACKTEST")
print("=" * 70)
print(f"{'Metric':<10}{'Local GBT':>14}{'7-Day Lag':>14}{'Improvement':>16}")
print("-" * 54)
print(f"{'MAE':<10}{local_mae:>14.2f}{baseline_mae:>14.2f}{baseline_mae-local_mae:>16.2f}")
print(f"{'RMSE':<10}{local_rmse:>14.2f}{baseline_rmse:>14.2f}{baseline_rmse-local_rmse:>16.2f}")
print(f"{'WAPE %':<10}{local_wape:>14.2f}{baseline_wape:>14.2f}{baseline_wape-local_wape:>16.2f}")


LOCAL ML MODEL COMPARISON — 14-DAY BACKTEST
Metric         Local GBT     7-Day Lag     Improvement
------------------------------------------------------
MAE                 1.10          1.73            0.63
RMSE                1.84          3.05            1.20
WAPE %             29.48         46.45           16.97


In [14]:
# ============================================================
# CELL 13 — COMPARE WITH THE EXISTING SPARK GBT RESULT
# ============================================================

# These are the metrics already obtained from the existing notebook.
SPARK_GBT_MAE = 1.12
SPARK_GBT_RMSE = 1.92
SPARK_GBT_WAPE = 30.09

print("=" * 70)
print("EXISTING SPARK GBT vs LOCAL GRADIENT BOOSTING")
print("=" * 70)

comparison = pd.DataFrame([
    {
        "model": "Existing Spark GBT",
        "MAE": SPARK_GBT_MAE,
        "RMSE": SPARK_GBT_RMSE,
        "WAPE_%": SPARK_GBT_WAPE,
    },
    {
        "model": "Local Gradient Boosting",
        "MAE": local_mae,
        "RMSE": local_rmse,
        "WAPE_%": local_wape,
    },
    {
        "model": "7-Day Lag Baseline",
        "MAE": baseline_mae,
        "RMSE": baseline_rmse,
        "WAPE_%": baseline_wape,
    },
])

print(comparison.to_string(index=False))

print(
    "\nThe local model will be used for recursive future inference "
    "only if its backtest is acceptable."
)


EXISTING SPARK GBT vs LOCAL GRADIENT BOOSTING
                  model      MAE     RMSE    WAPE_%
     Existing Spark GBT 1.120000 1.920000 30.090000
Local Gradient Boosting 1.097062 1.843892 29.479588
     7-Day Lag Baseline 1.728571 3.046309 46.449136

The local model will be used for recursive future inference only if its backtest is acceptable.


In [15]:
# ============================================================
# CELL 14 — PREPARE RECURSIVE FORECAST STATE
# ============================================================

FORECAST_DAYS = 7
FORECAST_START = max_date + timedelta(days=1)
FORECAST_END = max_date + timedelta(days=FORECAST_DAYS)

pairs_pdf = (
    history_pdf[["store_id", "product_id"]]
    .drop_duplicates()
    .sort_values(["store_id", "product_id"])
    .reset_index(drop=True)
)

assert len(pairs_pdf) == 50

# Demand history: exactly the latest 28 actual observations per series.
demand_history = {}

for (store_id, product_id), group in history_pdf.groupby(
    ["store_id", "product_id"],
    sort=False,
):
    values = (
        group.sort_values("date")["quantity_sold"]
        .astype(float)
        .tail(28)
        .tolist()
    )

    if len(values) != 28:
        raise ValueError(
            f"{store_id}/{product_id} has {len(values)} history rows; expected 28."
        )

    demand_history[(store_id, product_id)] = values

# Future exogenous values:
# use the latest known value for each store-product pair.
latest_rows = (
    history_pdf
    .sort_values("date")
    .groupby(["store_id", "product_id"], as_index=False)
    .tail(1)
)

latest_exogenous = {
    (row.store_id, row.product_id): {
        "promotion_applied": float(row.promotion_applied),
        "promotion_transactions": float(row.promotion_transactions),
        "inventory_min_before": float(row.inventory_min_before),
        "average_unit_price_cents": float(row.average_unit_price_cents),
    }
    for row in latest_rows.itertuples(index=False)
}

print("=" * 60)
print("RECURSIVE FORECAST SETUP")
print("=" * 60)
print("Last historical date:", max_date)
print("Forecast start      :", FORECAST_START)
print("Forecast end        :", FORECAST_END)
print("Pairs               :", len(pairs_pdf))
print("History per pair    : 28 observations")
print("Future exogenous    : latest-known-value assumption")


RECURSIVE FORECAST SETUP
Last historical date: 2026-08-12
Forecast start      : 2026-08-13
Forecast end        : 2026-08-19
Pairs               : 50
History per pair    : 28 observations
Future exogenous    : latest-known-value assumption


In [16]:
# ============================================================
# CELL 15 — RECURSIVE 7-DAY LOCAL ML FORECAST
# ============================================================

working_history = {
    key: list(values)
    for key, values in demand_history.items()
}

forecast_rows = []

for day_offset in range(FORECAST_DAYS):
    forecast_date = FORECAST_START + timedelta(days=day_offset)

    day_rows = []

    for row in pairs_pdf.itertuples(index=False):
        key = (row.store_id, row.product_id)
        values = working_history[key]

        if len(values) < 28:
            raise RuntimeError(
                f"Insufficient history for {key}: {len(values)} rows."
            )

        exog = latest_exogenous[key]

        # Recursive lag features.
        lag_1 = values[-1]
        lag_7 = values[-7]
        lag_14 = values[-14]
        lag_28 = values[-28]

        rolling_mean_7 = float(np.mean(values[-7:]))
        rolling_mean_14 = float(np.mean(values[-14:]))
        rolling_mean_28 = float(np.mean(values[-28:]))

        timestamp = pd.Timestamp(forecast_date)

        row_features = {
            "store_id": row.store_id,
            "product_id": row.product_id,
            "lag_1": lag_1,
            "lag_7": lag_7,
            "lag_14": lag_14,
            "lag_28": lag_28,
            "rolling_mean_7": rolling_mean_7,
            "rolling_mean_14": rolling_mean_14,
            "rolling_mean_28": rolling_mean_28,
            "promotion_applied": exog["promotion_applied"],
            "promotion_transactions": exog["promotion_transactions"],
            "inventory_min_before": exog["inventory_min_before"],
            "average_unit_price_cents": exog["average_unit_price_cents"],
            "day_of_week": timestamp.dayofweek,
            "day_of_month": timestamp.day,
            "month": timestamp.month,
        }

        day_rows.append(row_features)

    # Transform the complete 50-row day in one local operation.
    day_features_pdf = pd.DataFrame(day_rows)

    day_matrix = preprocessor.transform(
        day_features_pdf[MODEL_FEATURES]
    )

    day_predictions = np.maximum(
        local_model.predict(day_matrix),
        0.0,
    )

    for feature_row, prediction in zip(
        day_rows,
        day_predictions,
    ):
        key = (
            feature_row["store_id"],
            feature_row["product_id"],
        )

        predicted_qty = float(prediction)

        # Feed the prediction into the next day's lag history.
        working_history[key].append(predicted_qty)
        working_history[key] = working_history[key][-28:]

        forecast_rows.append({
            "date": forecast_date,
            "store_id": feature_row["store_id"],
            "product_id": feature_row["product_id"],
            "predicted_demand": predicted_qty,
        })

    print(
        f"{forecast_date} -> "
        f"{len(day_predictions)} predictions | "
        f"total = {day_predictions.sum():.2f}"
    )

print("Recursive local ML forecast completed.")


2026-08-13 -> 50 predictions | total = 176.39
2026-08-14 -> 50 predictions | total = 176.27
2026-08-15 -> 50 predictions | total = 180.39
2026-08-16 -> 50 predictions | total = 182.16
2026-08-17 -> 50 predictions | total = 169.77
2026-08-18 -> 50 predictions | total = 169.50
2026-08-19 -> 50 predictions | total = 170.34
Recursive local ML forecast completed.


In [17]:
# ============================================================
# CELL 16 — FORECAST VALIDATION
# ============================================================

forecast_pdf = pd.DataFrame(forecast_rows)

forecast_pdf["date"] = pd.to_datetime(
    forecast_pdf["date"]
).dt.date

forecast_rows_count = len(forecast_pdf)

unique_dates = forecast_pdf["date"].nunique()

unique_pairs = forecast_pdf[
    ["store_id", "product_id"]
].drop_duplicates().shape[0]

unique_grain = forecast_pdf[
    ["date", "store_id", "product_id"]
].drop_duplicates().shape[0]

print("=" * 60)
print("FORECAST VALIDATION")
print("=" * 60)
print("Rows                :", forecast_rows_count)
print("Distinct dates      :", unique_dates)
print("Store-product pairs :", unique_pairs)
print("Unique grain rows   :", unique_grain)
print("Date range          :", forecast_pdf["date"].min(), "->", forecast_pdf["date"].max())

assert forecast_rows_count == 350
assert unique_dates == 7
assert unique_pairs == 50
assert unique_grain == 350
assert forecast_pdf["date"].min() == FORECAST_START
assert forecast_pdf["date"].max() == FORECAST_END
assert forecast_pdf["predicted_demand"].notna().all()
assert (forecast_pdf["predicted_demand"] >= 0).all()

print("Forecast validation: PASSED")


FORECAST VALIDATION
Rows                : 350
Distinct dates      : 7
Store-product pairs : 50
Unique grain rows   : 350
Date range          : 2026-08-13 -> 2026-08-19
Forecast validation: PASSED


In [22]:
# ============================================================
# CELL 17 — SAMPLE FORECAST
# ============================================================

print("=" * 70)
print("7-DAY LOCAL ML FORECAST — SAMPLE")
print("=" * 70)

display(
    forecast_pdf
    .sort_values(["date", "store_id", "product_id"])
    .head(30)
)


7-DAY LOCAL ML FORECAST — SAMPLE


,date,store_id,product_id,predicted_demand
0,2026-08-13,STR001,PRD0001,6.667544
1,2026-08-13,STR001,PRD0002,3.669715
2,2026-08-13,STR001,PRD0003,1.061894
3,2026-08-13,STR001,PRD0004,1.113602
4,2026-08-13,STR001,PRD0005,2.454780
5,2026-08-13,STR001,PRD0006,3.630778
6,2026-08-13,STR001,PRD0007,1.061894
7,2026-08-13,STR001,PRD0008,1.061894
8,2026-08-13,STR001,PRD0009,4.596068
9,2026-08-13,STR001,PRD0010,7.075804


In [23]:
# ============================================================
# CELL 18 — DAILY FORECAST SUMMARY
# ============================================================

daily_summary = (
    forecast_pdf
    .groupby("date")
    .agg(
        active_pairs=("predicted_demand", "count"),
        total_predicted_units=("predicted_demand", "sum"),
        average_pair_velocity=("predicted_demand", "mean"),
    )
    .reset_index()
)

daily_summary["total_predicted_units"] = daily_summary[
    "total_predicted_units"
].round(2)

daily_summary["average_pair_velocity"] = daily_summary[
    "average_pair_velocity"
].round(2)

print("=" * 60)
print("DAILY TOTAL PREDICTED DEMAND")
print("=" * 60)

print(daily_summary.to_string(index=False))

assert (daily_summary["active_pairs"] == 50).all()


DAILY TOTAL PREDICTED DEMAND
      date  active_pairs  total_predicted_units  average_pair_velocity
2026-08-13            50                 176.39                   3.53
2026-08-14            50                 176.27                   3.53
2026-08-15            50                 180.39                   3.61
2026-08-16            50                 182.16                   3.64
2026-08-17            50                 169.77                   3.40
2026-08-18            50                 169.50                   3.39
2026-08-19            50                 170.34                   3.41


In [18]:
# ============================================================
# CELL 20 — PRODUCT-LEVEL SUMMARY
# ============================================================

product_summary = (
    forecast_pdf
    .groupby("product_id")
    .agg(
        cumulative_product_demand=("predicted_demand", "sum"),
        daily_mean_velocity=("predicted_demand", "mean"),
        peak_demand_spike=("predicted_demand", "max"),
    )
    .reset_index()
    .sort_values("cumulative_product_demand", ascending=False)
)

for col in [
    "cumulative_product_demand",
    "daily_mean_velocity",
    "peak_demand_spike",
]:
    product_summary[col] = product_summary[col].round(2)

print("=" * 60)
print("PRODUCT-LEVEL FORECAST SUMMARY — TOP 10")
print("=" * 60)

print(product_summary.head(10).to_string(index=False))


PRODUCT-LEVEL FORECAST SUMMARY — TOP 10
product_id  cumulative_product_demand  daily_mean_velocity  peak_demand_spike
   PRD0010                     258.78                 7.39               9.25
   PRD0001                     257.94                 7.37               8.93
   PRD0002                     218.85                 6.25               7.70
   PRD0006                     133.00                 3.80               4.96
   PRD0009                     125.07                 3.57               4.88
   PRD0005                      79.79                 2.28               2.62
   PRD0004                      37.98                 1.09               1.14
   PRD0007                      37.92                 1.08               1.13
   PRD0008                      37.90                 1.08               1.12
   PRD0003                      37.59                 1.07               1.13


In [25]:
# ============================================================
# CELL 21 — FORECAST SANITY CHECKS
# ============================================================

total_forecast = float(forecast_pdf["predicted_demand"].sum())
daily_totals = daily_summary["total_predicted_units"]

print("=" * 60)
print("FORECAST SANITY CHECK")
print("=" * 60)
print(f"7-day total predicted demand : {total_forecast:.2f}")
print(f"Minimum prediction            : {forecast_pdf['predicted_demand'].min():.4f}")
print(f"Maximum prediction            : {forecast_pdf['predicted_demand'].max():.4f}")
print(f"Mean prediction               : {forecast_pdf['predicted_demand'].mean():.4f}")

assert np.isfinite(total_forecast)
assert total_forecast >= 0
assert np.isfinite(daily_totals).all()

print("Sanity checks: PASSED")


FORECAST SANITY CHECK
7-day total predicted demand : 1224.82
Minimum prediction            : 1.0549
Maximum prediction            : 9.2489
Mean prediction               : 3.4995
Sanity checks: PASSED


In [19]:
# ============================================================
# CELL 23 — FINAL SIGN-OFF
# ============================================================

print("=" * 70)
print("RETAILFLOW LOCAL ML DEMAND FORECAST — FINAL STATUS")
print("=" * 70)
print("Historical data              : 9,000 rows")
print("Store-product pairs          : 50")
print("Forecast horizon              : 7 days")
print("Expected forecast rows       : 350")
print("Actual forecast rows         :", len(forecast_pdf))
print("Forecast range               :", FORECAST_START, "->", FORECAST_END)
print("ML inference engine          : scikit-learn GradientBoostingRegressor")
print("Recursive inference          : Python/local memory")
print("Spark ML recursive inference : NOT USED")
print("Future exogenous assumption  : latest known value")
print("=" * 70)
print("FINAL FORECAST STATUS: PASSED")
print("=" * 70)


RETAILFLOW LOCAL ML DEMAND FORECAST — FINAL STATUS
Historical data              : 9,000 rows
Store-product pairs          : 50
Forecast horizon              : 7 days
Expected forecast rows       : 350
Actual forecast rows         : 350
Forecast range               : 2026-08-13 -> 2026-08-19
ML inference engine          : scikit-learn GradientBoostingRegressor
Recursive inference          : Python/local memory
Spark ML recursive inference : NOT USED
Future exogenous assumption  : latest known value
FINAL FORECAST STATUS: PASSED


In [20]:
# ============================================================
# INVENTORY PHASE — CELL 1
# GOLD DATASET DISCOVERY
# ============================================================

from pyspark.sql import functions as F

GOLD_PATH = "s3a://retailflow/gold"

print("=" * 70)
print("RETAILFLOW GOLD DATASET AUDIT")
print("=" * 70)
print("Gold path:", GOLD_PATH)
print()

jvm = spark.sparkContext._jvm
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

gold_path = jvm.org.apache.hadoop.fs.Path(GOLD_PATH)
fs = gold_path.getFileSystem(hadoop_conf)

items = fs.listStatus(gold_path)

print("Gold datasets / directories:")
print("-" * 70)

for item in sorted(items, key=lambda x: x.getPath().toString()):
    print(item.getPath().toString())

print()
print("Gold dataset discovery: COMPLETED")

RETAILFLOW GOLD DATASET AUDIT
Gold path: s3a://retailflow/gold

Gold datasets / directories:
----------------------------------------------------------------------
s3a://retailflow/gold/category_sales
s3a://retailflow/gold/city_sales
s3a://retailflow/gold/customer_segments
s3a://retailflow/gold/daily_demand
s3a://retailflow/gold/daily_sales
s3a://retailflow/gold/financial_summary
s3a://retailflow/gold/forecasting_dataset
s3a://retailflow/gold/loyalty_analysis
s3a://retailflow/gold/payment_finance
s3a://retailflow/gold/payment_summary
s3a://retailflow/gold/store_sales
s3a://retailflow/gold/top_products

Gold dataset discovery: COMPLETED


In [21]:
# ============================================================
# INVENTORY / BUSINESS OUTPUTS — CELL 2
# GOLD SCHEMA AUDIT
# ============================================================

GOLD_DATASETS = {
    "daily_sales": "s3a://retailflow/gold/daily_sales",
    "forecasting_dataset": "s3a://retailflow/gold/forecasting_dataset",
    "financial_summary": "s3a://retailflow/gold/financial_summary",
    "payment_finance": "s3a://retailflow/gold/payment_finance",
}

for name, path in GOLD_DATASETS.items():

    print("\n" + "=" * 80)
    print(f"GOLD DATASET: {name}")
    print("=" * 80)
    print("Path:", path)

    df = (
        spark.read
        .format("delta")
        .load(path)
    )

    print("\nColumns:")
    for field in df.schema.fields:
        print(
            f"  {field.name:<35} "
            f"{field.dataType}"
        )

    print("\nRow count:", df.count())

    print("\nSample:")
    df.show(5, truncate=False)


GOLD DATASET: daily_sales
Path: s3a://retailflow/gold/daily_sales

Columns:
  transaction_timestamp               StringType()
  revenue                             LongType()
  transactions                        LongType()
  average_order_value                 DoubleType()

Row count: 14428

Sample:
+--------------------------+--------+------------+-------------------+
|transaction_timestamp     |revenue |transactions|average_order_value|
+--------------------------+--------+------------+-------------------+
|2026-05-24 21:27:06.936142|9450    |3           |3150.0             |
|2026-04-27 19:18:08.523888|17752797|3           |5917599.0          |
|2026-04-16 10:02:35.065314|42840   |3           |14280.0            |
|2026-07-26 17:04:56.37933 |21420   |3           |7140.0             |
|2026-04-22 09:45:34.259826|382725  |3           |127575.0           |
+--------------------------+--------+------------+-------------------+
only showing top 5 rows


GOLD DATASET: forecasting_datas

In [22]:
# ============================================================
# INVENTORY PHASE — CELL 3
# SILVER DATASET DISCOVERY
# ============================================================

SILVER_PATH = "s3a://retailflow/silver"

print("=" * 70)
print("RETAILFLOW SILVER DATASET AUDIT")
print("=" * 70)
print("Silver path:", SILVER_PATH)
print()

jvm = spark.sparkContext._jvm
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

silver_path = jvm.org.apache.hadoop.fs.Path(SILVER_PATH)
fs = silver_path.getFileSystem(hadoop_conf)

items = fs.listStatus(silver_path)

print("Silver datasets / directories:")
print("-" * 70)

for item in sorted(items, key=lambda x: x.getPath().toString()):
    print(item.getPath().toString())

print()
print("Silver dataset discovery: COMPLETED")

RETAILFLOW SILVER DATASET AUDIT
Silver path: s3a://retailflow/silver

Silver datasets / directories:
----------------------------------------------------------------------
s3a://retailflow/silver/transactions

Silver dataset discovery: COMPLETED


In [23]:
# ============================================================
# INVENTORY PHASE — CELL 4
# SILVER TRANSACTION SCHEMA AUDIT
# ============================================================

SILVER_TRANSACTIONS_PATH = "s3a://retailflow/silver/transactions"

print("=" * 70)
print("SILVER TRANSACTIONS AUDIT")
print("=" * 70)
print("Path:", SILVER_TRANSACTIONS_PATH)

silver_transactions_df = (
    spark.read
    .format("delta")
    .load(SILVER_TRANSACTIONS_PATH)
)

print("\nSCHEMA")
print("-" * 70)
silver_transactions_df.printSchema()

print("\nROW COUNT")
print("-" * 70)

silver_count = silver_transactions_df.count()
print("Rows:", f"{silver_count:,}")

print("\nSAMPLE DATA")
print("-" * 70)

silver_transactions_df.show(
    10,
    truncate=False
)

print("\nCOLUMNS")
print("-" * 70)

for column in silver_transactions_df.columns:
    print(" -", column)

print("\nSilver transaction audit: COMPLETED")

SILVER TRANSACTIONS AUDIT
Path: s3a://retailflow/silver/transactions

SCHEMA
----------------------------------------------------------------------
root
 |-- transaction_id: string (nullable = true)
 |-- transaction_timestamp: string (nullable = true)
 |-- invoice_number: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- cashier_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- loyalty_member: boolean (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- quantity: integer (nullable = 

In [24]:
# ============================================================
# INVENTORY PHASE — CELL 5
# INVENTORY EVENT VALIDATION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=" * 70)
print("INVENTORY EVENT VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check required inventory columns
# ------------------------------------------------------------

REQUIRED_COLUMNS = [
    "transaction_id",
    "transaction_timestamp",
    "store_id",
    "product_id",
    "quantity",
    "inventory_before",
    "inventory_after",
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in silver_transactions_df.columns
]

print("Required inventory columns:")
for column in REQUIRED_COLUMNS:
    print(f" - {column}")

print()

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

print("Required columns: PASSED")


# ------------------------------------------------------------
# 2. Check transaction_id uniqueness
# ------------------------------------------------------------

total_rows = silver_transactions_df.count()

distinct_transactions = (
    silver_transactions_df
    .select("transaction_id")
    .distinct()
    .count()
)

print()
print("Transaction grain:")
print("Total rows              :", total_rows)
print("Distinct transaction_id :", distinct_transactions)

assert total_rows == distinct_transactions, (
    "Duplicate transaction_id values detected."
)

print("Transaction uniqueness: PASSED")


# ------------------------------------------------------------
# 3. Check inventory arithmetic
# ------------------------------------------------------------
# For a sale:
#
# inventory_after should equal:
#
# inventory_before - quantity
#
# This is a critical business rule for the current POS model.

inventory_mismatch_df = (
    silver_transactions_df
    .filter(
        F.col("inventory_before").isNotNull()
        & F.col("inventory_after").isNotNull()
        & F.col("quantity").isNotNull()
    )
    .filter(
        F.col("inventory_after")
        != F.col("inventory_before") - F.col("quantity")
    )
)

inventory_mismatch_count = inventory_mismatch_df.count()

print()
print("Inventory arithmetic mismatches:", inventory_mismatch_count)

if inventory_mismatch_count > 0:
    print()
    print("First mismatches:")
    inventory_mismatch_df.select(
        "transaction_id",
        "store_id",
        "product_id",
        "quantity",
        "inventory_before",
        "inventory_after",
    ).show(10, truncate=False)

assert inventory_mismatch_count == 0, (
    "Inventory arithmetic validation failed."
)

print("Inventory arithmetic: PASSED")


# ------------------------------------------------------------
# 4. Check invalid negative inventory
# ------------------------------------------------------------

negative_inventory_count = (
    silver_transactions_df
    .filter(
        (F.col("inventory_before") < 0)
        | (F.col("inventory_after") < 0)
    )
    .count()
)

print()
print("Negative inventory records:", negative_inventory_count)

assert negative_inventory_count == 0, (
    "Negative inventory values detected."
)

print("Non-negative inventory: PASSED")


# ------------------------------------------------------------
# 5. Check store-product coverage
# ------------------------------------------------------------

distinct_pairs = (
    silver_transactions_df
    .select("store_id", "product_id")
    .distinct()
    .count()
)

print()
print("Distinct store-product pairs:", distinct_pairs)

print()
print("=" * 70)
print("INVENTORY EVENT VALIDATION COMPLETED")
print("=" * 70)

INVENTORY EVENT VALIDATION
Required inventory columns:
 - transaction_id
 - transaction_timestamp
 - store_id
 - product_id
 - quantity
 - inventory_before
 - inventory_after

Required columns: PASSED

Transaction grain:
Total rows              : 60051
Distinct transaction_id : 20017


AssertionError: Duplicate transaction_id values detected.

In [25]:
# ============================================================
# INVENTORY PHASE — CELL 6
# DUPLICATE TRANSACTION DISTRIBUTION
# ============================================================

from pyspark.sql import functions as F

print("=" * 70)
print("DUPLICATE TRANSACTION ANALYSIS")
print("=" * 70)

duplicate_groups = (
    silver_transactions_df
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate transaction IDs:", duplicate_groups.count())

print("\nDuplicate frequency distribution:")
(
    duplicate_groups
    .groupBy("count")
    .count()
    .orderBy("count")
    .show(truncate=False)
)

print("\nSample duplicated transaction IDs:")
(
    duplicate_groups
    .orderBy(F.desc("count"))
    .show(20, truncate=False)
)

DUPLICATE TRANSACTION ANALYSIS
Duplicate transaction IDs: 20017

Duplicate frequency distribution:
+-----+-----+
|count|count|
+-----+-----+
|3    |20017|
+-----+-----+


Sample duplicated transaction IDs:
+------------------------------------+-----+
|transaction_id                      |count|
+------------------------------------+-----+
|00d2641f-c492-4fb3-8dcb-039a8e69e9d0|3    |
|00e3e109-9385-4948-a111-0a47876c30d5|3    |
|0205b95e-ddcb-40f2-9b21-bb32b60295fa|3    |
|0310cd4c-ebfd-4f4e-8e8a-a6b75a5a3fa1|3    |
|060b4315-31b5-4811-9ed4-e96295083c6b|3    |
|04cd1804-23c0-42cc-bd63-106be7329de5|3    |
|06215393-5c24-45d8-bd7b-a7116012b985|3    |
|0654d82f-31a1-4607-a9c7-2e3c61cada54|3    |
|0b4d8509-6365-45b0-8406-f145847430c1|3    |
|08a6c4a6-504d-49ee-a3bb-c0cfb4f4b374|3    |
|0fbf660c-8b81-46f9-8c5d-34c5423999f3|3    |
|092ce0eb-c437-4131-bc6b-10d64083f315|3    |
|13c5ef63-a9a5-491e-8bbb-626b1c4b3bd5|3    |
|0a1f60ac-e92d-4e68-a7a9-ad21a67e7039|3    |
|15c3cb79-81ee-444e-9a48-3e7f

In [26]:
# ============================================================
# INVENTORY PHASE — CELL 7
# DUPLICATE CONTENT CONSISTENCY
# ============================================================

print("=" * 70)
print("DUPLICATE CONTENT CONSISTENCY")
print("=" * 70)

sample_duplicate_id = (
    silver_transactions_df
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .select("transaction_id")
    .first()["transaction_id"]
)

print("Sample transaction_id:", sample_duplicate_id)

sample_rows = (
    silver_transactions_df
    .filter(F.col("transaction_id") == sample_duplicate_id)
    .orderBy("created_at")
)

sample_rows.show(
    10,
    truncate=False
)

print("\nRows for sample transaction:")
print(sample_rows.count())

print("\nDistinct values across important fields:")

CHECK_COLUMNS = [
    "transaction_id",
    "transaction_timestamp",
    "store_id",
    "product_id",
    "quantity",
    "inventory_before",
    "inventory_after",
    "created_at",
]

for column in CHECK_COLUMNS:
    distinct_count = (
        sample_rows
        .select(column)
        .distinct()
        .count()
    )

    print(f"{column:<25}: {distinct_count}")

DUPLICATE CONTENT CONSISTENCY
Sample transaction_id: 00d2641f-c492-4fb3-8dcb-039a8e69e9d0
+------------------------------------+--------------------------+--------------+--------+-----------------+-------+------+------+-----------+----------+-----------+----------------+--------------+----------+-------------+---------+-----------+------+-----------+--------+----------------+--------------+---------+------------------+--------+--------------+----------------+------------+----------------+---------------+--------------------------+
|transaction_id                      |transaction_timestamp     |invoice_number|store_id|store_name       |country|city  |region|terminal_id|cashier_id|customer_id|customer_segment|loyalty_member|product_id|product_name |category |subcategory|brand |supplier_id|quantity|unit_price_cents|discount_cents|tax_cents|total_amount_cents|currency|payment_method|payment_provider|promotion_id|inventory_before|inventory_after|created_at                |
+---------------

In [27]:
# ============================================================
# INVENTORY PHASE — CELL 8
# SILVER DELTA HISTORY
# ============================================================

from delta.tables import DeltaTable

SILVER_TRANSACTIONS_PATH = "s3a://retailflow/silver/transactions"

print("=" * 70)
print("SILVER DELTA HISTORY")
print("=" * 70)

silver_delta = DeltaTable.forPath(
    spark,
    SILVER_TRANSACTIONS_PATH,
)

history_df = silver_delta.history()

print("Delta versions:")
history_df.select(
    "version",
    "timestamp",
    "operation",
    "operationParameters",
).show(20, truncate=False)

SILVER DELTA HISTORY
Delta versions:
+-------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                                                                                                                                          |
+-------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|74     |2026-08-17 17:32:01|WRITE    |{mode -> Append, partitionBy -> []}                                                                                                                                          |
|73     |2026-08-17 17:08:50|WRITE    |{mode -> Append, partitionBy -> []}                                 

In [28]:
# ============================================================
# INVENTORY PHASE — CELL 9
# REPAIR SILVER DUPLICATES
# ============================================================

from pyspark.sql import functions as F

SILVER_TRANSACTIONS_PATH = "s3a://retailflow/silver/transactions"

print("=" * 70)
print("SILVER DUPLICATE REPAIR")
print("=" * 70)

# ------------------------------------------------------------
# 1. Read current Silver
# ------------------------------------------------------------

silver_df = (
    spark.read
    .format("delta")
    .load(SILVER_TRANSACTIONS_PATH)
)

before_count = silver_df.count()

print("Current Silver rows :", before_count)

# ------------------------------------------------------------
# 2. Remove exact transaction duplicates
# ------------------------------------------------------------
# transaction_id is the business identifier for one transaction.
#
# We already verified that every duplicated transaction has
# identical business fields, so keeping one record per
# transaction_id is safe for the current dataset.

deduplicated_df = (
    silver_df
    .dropDuplicates(["transaction_id"])
)

after_count = deduplicated_df.count()

print("Deduplicated rows    :", after_count)
print("Rows removed         :", before_count - after_count)

# ------------------------------------------------------------
# 3. Safety checks BEFORE writing
# ------------------------------------------------------------

assert before_count == 60051, (
    f"Unexpected source row count: {before_count}"
)

assert after_count == 20017, (
    f"Expected 20,017 unique transactions, got {after_count}"
)

assert (
    deduplicated_df
    .select("transaction_id")
    .distinct()
    .count()
    == after_count
)

print()
print("Deduplication validation: PASSED")

# ------------------------------------------------------------
# 4. Replace Silver with corrected dataset
# ------------------------------------------------------------

(
    deduplicated_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_TRANSACTIONS_PATH)
)

print()
print("Silver table rewritten successfully.")

# ------------------------------------------------------------
# 5. Verify the actual Delta table after rewrite
# ------------------------------------------------------------

verified_df = (
    spark.read
    .format("delta")
    .load(SILVER_TRANSACTIONS_PATH)
)

verified_count = verified_df.count()

verified_distinct = (
    verified_df
    .select("transaction_id")
    .distinct()
    .count()
)

print()
print("=" * 70)
print("POST-REPAIR VERIFICATION")
print("=" * 70)
print("Rows                  :", verified_count)
print("Distinct transactions :", verified_distinct)

assert verified_count == 20017
assert verified_distinct == 20017

print()
print("🎉 SILVER DEDUPLICATION: PASSED")

SILVER DUPLICATE REPAIR
Current Silver rows : 60051
Deduplicated rows    : 20017
Rows removed         : 40034

Deduplication validation: PASSED

Silver table rewritten successfully.

POST-REPAIR VERIFICATION
Rows                  : 20017
Distinct transactions : 20017

🎉 SILVER DEDUPLICATION: PASSED


In [29]:
from pyspark.sql import functions as F

SILVER_TRANSACTIONS_PATH = "s3a://retailflow/silver/transactions"

silver_transactions_df = (
    spark.read
    .format("delta")
    .load(SILVER_TRANSACTIONS_PATH)
)

inventory_check = (
    silver_transactions_df
    .filter(
        F.col("inventory_before").isNotNull()
        & F.col("inventory_after").isNotNull()
        & F.col("quantity").isNotNull()
    )
    .withColumn(
        "expected_inventory_after",
        F.col("inventory_before") - F.col("quantity")
    )
    .withColumn(
        "inventory_difference",
        F.col("inventory_after") - F.col("expected_inventory_after")
    )
)

mismatch_count = (
    inventory_check
    .filter(F.col("inventory_difference") != 0)
    .count()
)

negative_count = (
    inventory_check
    .filter(
        (F.col("inventory_before") < 0)
        | (F.col("inventory_after") < 0)
    )
    .count()
)

print("Silver rows:", silver_transactions_df.count())
print("Inventory arithmetic mismatches:", mismatch_count)
print("Negative inventory records:", negative_count)

if mismatch_count > 0:
    print("\nSample mismatches:")
    (
        inventory_check
        .filter(F.col("inventory_difference") != 0)
        .select(
            "transaction_id",
            "store_id",
            "product_id",
            "quantity",
            "inventory_before",
            "inventory_after",
            "expected_inventory_after",
            "inventory_difference"
        )
        .show(10, truncate=False)
    )

assert mismatch_count == 0, "Inventory arithmetic validation failed"
assert negative_count == 0, "Negative inventory detected"

print("\nInventory arithmetic validation: PASSED")

Silver rows: 20017
Inventory arithmetic mismatches: 0
Negative inventory records: 0

Inventory arithmetic validation: PASSED


In [30]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_TRANSACTIONS_PATH = "s3a://retailflow/silver/transactions"
INVENTORY_GOLD_PATH = "s3a://retailflow/gold/inventory_current"

silver_transactions_df = (
    spark.read
    .format("delta")
    .load(SILVER_TRANSACTIONS_PATH)
)

inventory_events_df = (
    silver_transactions_df
    .withColumn(
        "event_timestamp",
        F.to_timestamp("transaction_timestamp")
    )
)

assert (
    inventory_events_df
    .filter(F.col("event_timestamp").isNull())
    .count()
    == 0
), "Invalid transaction timestamps detected"

latest_window = (
    Window
    .partitionBy("store_id", "product_id")
    .orderBy(
        F.col("event_timestamp").desc(),
        F.col("transaction_id").desc()
    )
)

inventory_current_df = (
    inventory_events_df
    .withColumn(
        "row_number",
        F.row_number().over(latest_window)
    )
    .filter(F.col("row_number") == 1)
    .select(
        "store_id",
        "store_name",
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "supplier_id",
        "inventory_after",
        "event_timestamp",
        "transaction_id"
    )
    .withColumnRenamed(
        "inventory_after",
        "inventory_quantity"
    )
    .withColumnRenamed(
        "event_timestamp",
        "last_transaction_timestamp"
    )
    .withColumn(
        "inventory_status",
        F.when(
            F.col("inventory_quantity") == 0,
            F.lit("OUT_OF_STOCK")
        )
        .when(
            F.col("inventory_quantity") <= 10,
            F.lit("LOW_STOCK")
        )
        .otherwise(
            F.lit("IN_STOCK")
        )
    )
)

inventory_current_df = inventory_current_df.cache()

inventory_rows = inventory_current_df.count()
inventory_pairs = (
    inventory_current_df
    .select("store_id", "product_id")
    .distinct()
    .count()
)

print("Inventory rows:", inventory_rows)
print("Store-product pairs:", inventory_pairs)

inventory_current_df.orderBy(
    "store_id",
    "product_id"
).show(20, truncate=False)

assert inventory_pairs == 50, (
    f"Expected 50 store-product pairs, got {inventory_pairs}"
)

assert inventory_rows == 50, (
    f"Expected 50 inventory records, got {inventory_rows}"
)

assert (
    inventory_current_df
    .filter(F.col("inventory_quantity") < 0)
    .count()
    == 0
), "Negative current inventory detected"

print("Inventory current-state validation: PASSED")

Inventory rows: 50
Store-product pairs: 50
+--------+--------------------------+----------+------------------------+-----------+-----------+-------+-----------+------------------+--------------------------+------------------------------------+----------------+
|store_id|store_name                |product_id|product_name            |category   |subcategory|brand  |supplier_id|inventory_quantity|last_transaction_timestamp|transaction_id                      |inventory_status|
+--------+--------------------------+----------+------------------------+-----------+-----------+-------+-----------+------------------+--------------------------+------------------------------------+----------------+
|STR001  |RetailFlow Connaught Place|PRD0001   |Amul Full Cream Milk 1L |Groceries  |Dairy      |Amul   |SUP001     |86                |2026-08-17 10:23:19.133027|7b532eae-0573-4505-b63a-c62ae7bd8aaa|IN_STOCK        |
|STR001  |RetailFlow Connaught Place|PRD0002   |Fortune Basmati Rice 5kg|Groceries  |

In [31]:
INVENTORY_GOLD_PATH = "s3a://retailflow/gold/inventory_current"

(
    inventory_current_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(INVENTORY_GOLD_PATH)
)

print("Gold inventory table written.")
print("Path:", INVENTORY_GOLD_PATH)

Gold inventory table written.
Path: s3a://retailflow/gold/inventory_current


In [32]:
INVENTORY_GOLD_PATH = "s3a://retailflow/gold/inventory_current"

inventory_gold_df = (
    spark.read
    .format("delta")
    .load(INVENTORY_GOLD_PATH)
)

row_count = inventory_gold_df.count()

pair_count = (
    inventory_gold_df
    .select("store_id", "product_id")
    .distinct()
    .count()
)

duplicate_pairs = (
    inventory_gold_df
    .groupBy("store_id", "product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

negative_inventory = (
    inventory_gold_df
    .filter(F.col("inventory_quantity") < 0)
    .count()
)

print("Gold rows:", row_count)
print("Store-product pairs:", pair_count)
print("Duplicate pairs:", duplicate_pairs)
print("Negative inventory:", negative_inventory)

inventory_gold_df.orderBy(
    "store_id",
    "product_id"
).show(50, truncate=False)

assert row_count == 50
assert pair_count == 50
assert duplicate_pairs == 0
assert negative_inventory == 0

print("\nGold inventory verification: PASSED")

Gold rows: 50
Store-product pairs: 50
Duplicate pairs: 0
Negative inventory: 0
+--------+--------------------------+----------+------------------------+-----------+-----------+-------+-----------+------------------+--------------------------+------------------------------------+----------------+
|store_id|store_name                |product_id|product_name            |category   |subcategory|brand  |supplier_id|inventory_quantity|last_transaction_timestamp|transaction_id                      |inventory_status|
+--------+--------------------------+----------+------------------------+-----------+-----------+-------+-----------+------------------+--------------------------+------------------------------------+----------------+
|STR001  |RetailFlow Connaught Place|PRD0001   |Amul Full Cream Milk 1L |Groceries  |Dairy      |Amul   |SUP001     |86                |2026-08-17 10:23:19.133027|7b532eae-0573-4505-b63a-c62ae7bd8aaa|IN_STOCK        |
|STR001  |RetailFlow Connaught Place|PRD0002   |F

In [33]:
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

silver_dir = project_root / "pipelines" / "silver"

print("Project root:", project_root)
print("Silver directory:", silver_dir)
print()

if not silver_dir.exists():
    raise FileNotFoundError(f"Silver directory not found: {silver_dir}")

for file_path in sorted(silver_dir.rglob("*.py")):
    print(file_path.relative_to(project_root))

Project root: c:\RetailFlow
Silver directory: c:\RetailFlow\pipelines\silver

pipelines\silver\silver_pipeline.py
pipelines\silver\validator.py
